In [ ]:
'''
Imports
'''

import os
import gc
import time
import re
import math
import tqdm
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import MinkowskiEngine as ME
from scipy.special import expit
from glob import glob
from torch.utils.data import Subset, Dataset, DataLoader
import sys
import random

from modules import print_slice2, collate_sparse_minkowski
from dataset import SparseEvent
from models import minkunet
from sklearn.metrics import classification_report, confusion_matrix, explained_variance_score, accuracy_score
from combined_scheduler import CustomLambdaLR, CombinedScheduler

# manually specify the GPUs to use
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Define pixel and signal ranges
ranges = [-30, 350]  # Max pixels 64, signal loss 2.08
source_range = [-1, 1]

prefix = "small_iso"
dataset_path = "/scratch/salonso/sparse-nns/medical_data/data_kits23_roi_iso_{}_{}_good_pad15/*"
dataset = SparseEvent(dataset_path.format(ranges[0], ranges[1]))

In [ ]:
event = dataset[0]
c, x, y, width, height, nslices = event["c"], event["x"], event["y"], event["width"], event["height"], event["nslices"]
print_slice2(c, x, y, slice=200, height=height, width=width, nslices=nslices)

In [ ]:
'''
Calculate N folds
'''

np.random.seed(42)  # for reproducibility
indexes = np.arange(len(dataset))  # dataset indexes
np.random.shuffle(indexes)  # shuffle indexes

# Number of folds
folds = 5
train_indexes_folds = []
test_indexes_folds = []

for fold in range(folds):
    # Calculate start and end for training indexes
    train_start = int((fold * 100 / folds) * (len(indexes) / 100))
    train_end = int(((fold * 100 / folds + (100 - 10 / folds * 10)) % 100) * (len(indexes) / 100))
    
    if train_end < train_start:
        train_indexes = np.concatenate([indexes[train_start:], indexes[:train_end]])
    else:
        train_indexes = indexes[train_start:train_end]
    
    # Calculate start and end for test indexes
    test_start = train_end
    test_end = int(((fold * 100 / folds + 100) % 100) * (len(indexes) / 100))
    
    if test_end < test_start:
        test_indexes = indexes[test_start:]
    else:
        test_indexes = indexes[test_start:test_end]
    
    train_indexes_folds.append(train_indexes)
    test_indexes_folds.append(test_indexes)
    
    print(f"Fold {fold + 1}")
    print(f"Train indexes: {train_start} to {train_end}")
    print(f"Test indexes: {test_start} to {test_end}")
    print("====")


In [ ]:
'''
train/val/test sets and loaders for K folds
'''

batch_size = 1
num_workers = 8

collate = collate_sparse_minkowski

train_sets = [Subset(dataset, train_indexes) for train_indexes in train_indexes_folds]
test_sets = [Subset(dataset, test_indexes) for test_indexes in test_indexes_folds] 

train_loaders = [DataLoader(train_set, collate_fn=collate, batch_size=batch_size,\
                            num_workers=num_workers, shuffle=True) for train_set in train_sets]
test_loaders = [DataLoader(test_set, collate_fn=collate, batch_size=batch_size,\
                            num_workers=num_workers, shuffle=False) for test_set in test_sets]

In [ ]:
'''
Train and evaluation functions
'''

def arrange_sparse_minkowski(data):
    return ME.SparseTensor(
        features=data['f'].reshape(-1, 1),
        coordinates=ME.utils.batched_coordinates(data['c'], dtype=torch.int),
        device=device
    )

def arrange_truth(data, step=0):
    return (data[f'kidney_tumor_cyst{step}'], 
            data[f'tumor_cyst{step}'], 
            data[f'tumor_only{step}'])  
    
def train(loader, disable=False, gradient_accumulation_steps=1):
    
    model.train()
    dataset.train = True
    
    batch_size = loader.batch_size
    n_batches = int(math.ceil(len(loader.dataset)/batch_size)) #if max_iters_train is None else max_iters_train
    t = tqdm.tqdm(enumerate(loader),total=n_batches, disable=disable)
    sum_loss = 0.
    accumulated_gradients = 0
    
    optimizer.zero_grad()
    
    for i, data in t:
        if i%1==0:
            with torch.cuda.device(device):
                torch.cuda.empty_cache()

        batch_input = arrange_sparse_minkowski(data)
        batch_targetA, batch_targetB, batch_targetC = [], [], []
        for step in range(4):
            A, B, C = arrange_truth(data, int(2**step))
            batch_targetA.append(torch.reshape(A, (len(A), 1)).to(device))
            batch_targetB.append(torch.reshape(B, (len(B), 1)).to(device))
            batch_targetC.append(torch.reshape(C, (len(C), 1)).to(device))

        # Run model. batch_output = [output1, output2, output4, output8]
        batch_output = model(batch_input)

        # Calculate losses
        focal_loss_total, dice_loss_total = None, None
        for step in range(4):
            # Retrieve prediction and target
            pred = batch_output[step].F
            targetA = batch_targetA[step]
            targetB = batch_targetB[step]
            targetC = batch_targetC[step]

            print(pred.shape, targetA.shape, targetB.shape, targetC.shape)

            # Calculate sigmoid focal and dice loss
            focal_lossA = focal_loss_fn(pred[:, 0:1], targetA, alpha = -1, gamma = 2, reduction = "mean")
            focal_lossB = focal_loss_fn(pred[:, 1:2], targetB, alpha = -1, gamma = 2, reduction = "mean")
            focal_lossC = focal_loss_fn(pred[:, 2:3], targetC, alpha = -1, gamma = 2, reduction = "mean")
            dice_lossA = dice_loss_fn(pred[:, 0:1], targetA)
            dice_lossB = dice_loss_fn(pred[:, 1:2], targetB)
            dice_lossC = dice_loss_fn(pred[:, 2:3], targetC)

            focal_loss = focal_lossA + focal_lossB + focal_lossC
            dice_loss = dice_lossA + dice_lossB + dice_lossC

            # Total loss
            if step == 0:
                focal_loss_total = focal_loss
                dice_loss_total = dice_loss
            else:
                weight = 1/(2**step)
                focal_loss_total += weight*focal_loss
                dice_loss_total += weight*dice_loss

        # Total loss
        batch_loss = focal_loss_total + dice_loss_total
        batch_loss.backward()
        t.set_description("loss = {0:.5f}, focal_loss = {1:.5f}, dice_loss = {2:.5f}".format(batch_loss.item(), 
                                                                                             focal_loss_total.item(), 
                                                                                             dice_loss_total.item()) )
        # Sum loss
        sum_loss += batch_loss.item()
        accumulated_gradients += 1
        
        if accumulated_gradients % gradient_accumulation_steps == 0:
            # Gradient accumulation
            optimizer.step()
            scheduler.step()
            accumulated_gradients = 0
            optimizer.zero_grad()
            
    # Perform a final gradient update if there are remaining accumulated gradients
    if accumulated_gradients > 0:
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
          
    return sum_loss / n_batches

def test(loader, disable=False):
    
    model.eval()
    dataset.train = False
    
    batch_size = loader.batch_size
    n_batches = int(math.ceil(len(loader.dataset)/batch_size)) #if max_iters_train is None else max_iters_train
    t = tqdm.tqdm(enumerate(loader),total=n_batches, disable=disable)
    
    sum_loss, sum_loss_focal, sum_loss_dice = 0., 0., 0.
    for i, data in t:
        torch.cuda.empty_cache()

        batch_input = arrange_sparse_minkowski(data)
        batch_targetA, batch_targetB, batch_targetC = [], [], []
        for step in range(4):
            A, B, C = arrange_truth(data, int(2**step))
            batch_targetA.append(torch.reshape(A, (len(A), 1)).to(device))
            batch_targetB.append(torch.reshape(B, (len(B), 1)).to(device))
            batch_targetC.append(torch.reshape(C, (len(C), 1)).to(device))

        # Run model. batch_output = [output1, output2, output4, output8]
        batch_output = model(batch_input)

        # Calculate losses
        focal_loss_total, dice_loss_total = None, None
        for step in range(4):
            # Retrieve prediction and target
            pred = batch_output[step].F
            targetA = batch_targetA[step]
            targetB = batch_targetB[step]
            targetC = batch_targetC[step]
            
            # Calculate sigmoid focal and dice loss
            focal_lossA = focal_loss_fn(pred[:, 0:1], targetA, alpha = -1, gamma = 2, reduction = "mean")
            focal_lossB = focal_loss_fn(pred[:, 1:2], targetB, alpha = -1, gamma = 2, reduction = "mean")
            focal_lossC = focal_loss_fn(pred[:, 2:3], targetC, alpha = -1, gamma = 2, reduction = "mean")
            dice_lossA = dice_loss_fn(pred[:, 0:1], targetA)
            dice_lossB = dice_loss_fn(pred[:, 1:2], targetB)
            dice_lossC = dice_loss_fn(pred[:, 2:3], targetC)

            focal_loss = focal_lossA + focal_lossB + focal_lossC
            dice_loss = dice_lossA + dice_lossB + dice_lossC

            # Total loss
            if step == 0:
                focal_loss_total = focal_loss
                dice_loss_total = dice_loss
            else:
                weight = 1/(2**step)
                focal_loss_total += weight*focal_loss
                dice_loss_total += weight*dice_loss

        # Total loss
        batch_loss = focal_loss_total + dice_loss_total
        t.set_description("val_loss = {0:.5f}, val_focal_loss = {1:.5f}, val_dice_loss = {2:.5f}".format(batch_loss.item(), 
                                                                                                         focal_loss_total.item(), 
                                                                                                         dice_loss_total.item()) )
        # Sum loss
        sum_loss += batch_loss.item()
        sum_loss_focal += focal_loss_total.item()
        sum_loss_dice += dice_loss_total.item()
          
    return sum_loss / n_batches, sum_loss_focal / n_batches, sum_loss_dice / n_batches


In [ ]:
from warmup_scheduler_pytorch import WarmUpScheduler
import torchvision

random.seed(42)  # for reproducibility

def dice_loss_fn(pred, true, eps=1e-5):
    pred = torch.sigmoid(pred)
    intersection = torch.sum(true * pred)
    union = torch.sum(true) + torch.sum(pred)
    dice = (2.0 * intersection + eps) / (union + eps)
    return 1.0 - dice  # minimisation problem

num_classes = 1
disable = False
focal_loss_fn = torchvision.ops.focal_loss.sigmoid_focal_loss
lr = 0.005
batch_size_real = 1
gradient_accumulation_steps = batch_size_real // batch_size
save_path = "/scratch2/salonso/medical/models/"

def initialize_weights(model):
    for m in model.modules():
        if isinstance(m, ME.MinkowskiConvolution):
            torch.nn.init.kaiming_normal_(m.kernel, mode="fan_out", nonlinearity="relu")

for fold in range(len(train_loaders)):
    print("________________________________")
    print("Fold {}: Initialising model...".format(fold))
    model_name = "MinkUNet34A"
    if model_name=="MinkUNet14A":
        model = minkunet.MinkUNet14A(in_channels=1, out_channels=num_classes, D=3).to(device)
    elif model_name=="MinkUNet34A":
        model = minkunet.MinkUNet34A(in_channels=1, out_channels=num_classes, D=3).to(device)
    elif model_name=="MinkUNet34B":
        model = minkunet.MinkUNet34B(in_channels=1, out_channels=num_classes, D=3).to(device)

    initialize_weights(model) # initialise weights
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-9, weight_decay=0.01)
    num_cycles = 25
    num_warmup_steps = 1
    
    # Calculate arguments for scheduler
    warmup_steps = len(train_loaders[fold]) * num_warmup_steps // gradient_accumulation_steps
    cosine_annealing_steps = len(train_loaders[fold]) * num_cycles // gradient_accumulation_steps
    
    # Create a warm-up scheduler that gradually increases the learning rate
    warmup_scheduler = CustomLambdaLR(optimizer, warmup_steps)

    # Create a cosine annealing scheduler that reduces learning rate in a cosine-like manner
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=cosine_annealing_steps,
                                                                            T_mult=1, eta_min=0.)
    
    scheduler = CombinedScheduler(optimizer=optimizer,
                                  scheduler1=warmup_scheduler,
                                  scheduler2=cosine_scheduler,
                                  warmup_steps=warmup_steps,
                                  lr_decay=1.0)
    
    count=0

    print("Fold {}: Starting training...".format(fold))

    min_val_loss = np.inf
    
    #stats = [[], [], []]
    stats = {"epoch": [],
             "train_loss": [],
             "val_loss": [],
             "val_loss_focal": [],
             "val_loss_dice": [],
             "lr": [],
            }
    
    for epoch in range(0, num_cycles*1+num_warmup_steps):
        if count >= 30:
            print("Early stopping: finishing....")
            break

        t0 = time.time()  # time start

        loss = train(train_loaders[fold], disable, gradient_accumulation_steps=gradient_accumulation_steps)
        
        t1 = time.time()  # time after training

        val_loss, val_loss_focal, val_loss_dice = test(test_loaders[fold], disable)
        
        t2 = time.time()  # time after validation
        
        lr = optimizer.param_groups[0]['lr']

        print(f'  - Epoch: {epoch:03d}, train_loss: {loss:.4f}, lr: {lr:.6f}')
        print(f'                        val_loss: {val_loss:.4f}, val_loss_focal: {val_loss_focal:.4f}, val_loss_dice: {val_loss_dice:.4f}')

        stats["epoch"].append(epoch)
        stats["train_loss"].append(loss)
        stats["val_loss"].append(val_loss)
        stats["val_loss_focal"].append(val_loss_focal)
        stats["val_loss_dice"].append(val_loss_dice)
        stats["lr"].append(lr)

        if val_loss_dice < min_val_loss:
            print("Saving model with val loss (dice): {0:.4f}".format(val_loss_dice))
            torch.save({'model': model.state_dict(),
                        'stats': stats}, 
                        save_path + "kits23_focdice_{}_{}_{}_{}_{}_{}_may2024".format(epoch, prefix, fold, model_name, ranges[0], ranges[1]))
            min_val_loss = val_loss_dice

        print("  - Epoch time (training): {0:.2f} s".format(t1 - t0))
        print("  - Epoch time (validation): {0:.2f} s".format(t2 - t1))

        t3 = time.time()
        print("  - Total epoch time: {0:.2f} s".format(t3 - t0))
        print("  --------")
                 
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
for i in train_loaders[0]:
    break

In [ ]:
train_loaders[0]